**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/bruno-2017"], check=True)
    os.chdir("/content/encore/entries/bruno-2017")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Bruno, Frost & Humphries (2017) from CRCNS data set apl-1

**Paper.** Bruno AM, Frost WN, Humphries MD (2017). A spiral attractor network drives rhythmic
locomotion. *eLife* 6:e27342. [doi:10.7554/elife.27342](https://doi.org/10.7554/elife.27342). The
claims file was written from the version of record (PMC5546814): main text, Materials and methods
and all figure legends and supplement legends.

**Data set.** CRCNS apl-1, [doi:10.6080/K0SN074B](https://doi.org/10.6080/K0SN074B), streamed from
the public bucket `crcnsarchive` under `apl-1/live/`. Version: `unversioned`. CRCNS data sets carry
no versions (docs/archives.md), and the bucket holds one copy of the processed files (`live/`; the
`orig/` folder holds only one raw optical file). That copy is pinned by the MD5 sums printed in the
data check, which match the data set's `checksums.md5`. The paper cites the data set's DOI and no
version. Files read: FileTable.mat, SpikeData.mat, ExtracellularData.mat, Coordinates.mat and
diode_template.mat; the 6.9 GB raw optical archive is not needed.

**Method.** The claims file decides what is tested. Where it leaves a detail open, the detail is
taken from the paper's Methods or from its code (github.com/mdhumphries/AplysiaAttractorAnalysis at
commit e5f28e98), cited in `apl.py`. Two details recur in almost every claim: the width of the
Gaussian kernel and the rule for the number of PCs. Each is run under every reading the criteria
list. Where one reading has to be chosen, the adopted reading is `own/min80`: the median ISI of the
recording, after onset, as the kernel SD, and the smallest number of PCs reaching 80% of the
variance. This follows the order of the protocol: the Methods first ("at least 80% of the
variance"; the median ISI "in population", the recording), then the code (its width, the median ISI,
where the printed width cannot give the paper's counts). No single reading makes every claim that
depends on it hold; the Summary lists, for each claim whose outcome changes with the reading, the
result under each. Outcomes that rest on the choice are marked as assumed in the record.

**Revision after verification round 1.** The conclusions were restored (the first build left them
out); windows count as coalesced and recurrent at *at least* 90%, as the Methods say; the decoding of
fig6d was run under all four readings and the 90% models of fig6s2 with the full history grid; fig7gh
was run under all four readings, with participation under each width; fig7s2 was tested with a port
of the code's ensemble toolbox; fig1d, fig3a, fig4bc and fig8 gained the outputs the verifier asked
for.

**Revision after verification round 2.** Text the numbers did not support was corrected (the
dimension ranges of the printed widths, the consistency of fig4c's means, the exclusion rule in
fig7gh); the code's removal rule of fig3h under the adopted reading is now stated; a printout of the
decoding forecasts that diverge was added, with a test of their sensitivity to rounding. The record
proposes a change of fig6d's outcome to not reproduced, for a third verification round.

**Revision after verification round 3.** The verifier agreed with the change, and fig6d is now not
reproduced. Its conclusion, and fig6s2's, now give the numbers of this execution: the decoding is
not exactly the same in every execution (fig6d), and the earlier prose described another one.

| Claim | Tested or blocked |
|---|---|
| dataset-size, fig1d-slow-oscillation, fig1e-decaying-rate, fig2b-dimensions | tested |
| fig2ef-periodic-recurrence, fig2e-single-dominant-orbit, fig2s2-threshold-robust, fig3a-coalescence | tested |
| fig3c-same-manifold, fig3s1a-common-axes, fig3d-initial-conditions, fig3f-pairwise-similarity | tested |
| fig3h-perturbations-return, fig3i-same-manifold-after | tested |
| fig4bc-stable-spiral, fig4c-orbital-period, fig4s1a-contracting-dimensions, fig4s1b-rotation-dominant | tested |
| fig5b-period-changes, fig5c-no-relation-contraction | tested |
| fig6s1-no-p10-motorneurons, fig6d-decode-p10, fig6s2-dimensions-p10 | tested |
| fig7a-long-tailed-participation, fig7f-strongly-variable-neurons, halving-participation, fig7s1-rate-synchrony, fig7gh-robust-to-participation | tested |
| fig7s2-no-invariant-core | tested (Python port of the code's ensemble toolbox) |
| fig8-caudolateral | blocked: metadata insufficient (orientation of the array not given) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import resource
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

import apl

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()
D = apl.load()
recs, preps = D["recs"], D["preps"]
print(f"CRCNS data set {apl.DATASET}, version {apl.VERSION}; read {len(D['md5'])} files from {apl.BUCKET} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
print("MD5 of each file read, against the data set's checksums.md5:")
for name, h in D["md5"].items():
    print(f"  {name:24s} {h}  {'matches' if D['listed_md5'].get(name) == h else 'DIFFERS from ' + str(D['listed_md5'].get(name))}")
info = pd.DataFrame([{
    "prep": r["prep"], "bout": r["bout"], "file": r["file"], "Bout_field": r["Bout_field"],
    "neurons": int(np.unique(r["ids"]).size), "max_id": int(r["ids"].max()),
    "spikes": r["t"].size, "first_spike_s": r["t"].min().round(3), "last_spike_s": r["t"].max().round(2),
    "neurons_spiking_after_30s": int(np.unique(r["ids"][r["t"] >= 30]).size)} for r in recs])
print("\nRecordings per preparation:", info.groupby("prep").size().to_dict())
print(info.to_string(index=False))
pt = pd.DataFrame(preps)
pt["neurons_in_spikes"] = [int(info[info.prep == p].neurons.iloc[0]) for p in pt.prep]
pt["coord_points"] = [c["x"].size for c in D["coords"]]
pt["coord_file"] = [c["file"] for c in D["coords"]]
pt["coord_ganglion"] = [c["ganglion"] for c in D["coords"]]
print("\nFileTable.mat against the spike and coordinate files:")
print(pt.to_string(index=False))
print("\nNeurons per preparation: min", pt.n_neurons_table.min(), "max", pt.n_neurons_table.max(),
      "total", pt.n_neurons_table.sum(), "(description: 75 to 160, 1131 in total)")
print("Same neuron count in all three bouts of every preparation:",
      bool(info.groupby("prep").neurons.nunique().eq(1).all()))

The data set holds 30 recordings, three bouts in each of 10 preparations, in the order of
FileTable.mat, which the description says is the paper's numbering. Every neuron id from 1 to N is
present in every bout of its preparation, and every neuron spikes after 30 s. The spike files carry
no bout number (`Bout` is empty); the bout is read from the file name (da01, da02, da03). The
neuron counts of FileTable.mat match the spike files and the coordinate files. The MD5 sums printed
above are the version pinned (CRCNS data sets are not versioned).

In [ ]:
# Units, ranges, the clock of the stimulus, and recordings deposited twice.
rows = []
for r in recs:
    t = r["t"]
    pre = np.sum((t >= 25) & (t < 30)) / 5
    stim = np.sum((t >= 30) & (t < 32.5)) / 2.5
    post = np.sum((t >= 32.5) & (t < 37.5)) / 5
    rows.append({"file": r["file"], "pop_rate_25_30": round(pre, 1), "pop_rate_30_32.5": round(stim, 1),
                 "pop_rate_32.5_37.5": round(post, 1)})
clock = pd.DataFrame(rows)
print("Population spike rate (spikes/s) before, during and after the stimulus at 30 to 32.5 s:")
print(clock.to_string(index=False))
print("Recordings whose rate after 32.5 s exceeds that before 30 s:",
      int((clock["pop_rate_32.5_37.5"] > clock["pop_rate_25_30"]).sum()), "of", len(clock))
# P10 nerve recordings
rows = []
for q in D["p10"]:
    t = q["t"]
    rows.append({"file": q["file"], "prep": q["prep"], "bout": q["bout"], "spikes": t.size,
                 "ids": np.unique(q["ids"]).tolist(), "time_sorted": q["sorted"],
                 "range_s": f"{t.min():.2f} to {t.max():.2f}",
                 "min_interval_ms": round(np.diff(t).min() * 1000, 3),
                 "rate_25_30": round(np.sum((t >= 25) & (t < 30)) / 5, 1),
                 "rate_32.5_37.5": round(np.sum((t >= 32.5) & (t < 37.5)) / 5, 1)})
print("\nP10 recordings:")
print(pd.DataFrame(rows).to_string(index=False))
print("Smallest P10 interval as a sampling period: 1 /", round(1 / np.diff(D["p10"][0]["t"]).min(), 1), "Hz")
# duplicate recordings: identical spike trains between any two recordings
sig = {}
for r in recs:
    for i, tr in enumerate(apl.neuron_trains(r)):
        if tr.size > 5:
            sig.setdefault(tuple(np.round(tr, 4)), []).append((r["file"], i + 1))
dup = [v for v in sig.values() if len(v) > 1]
print("\nNeuron spike trains (more than 5 spikes) found identical in two places:", len(dup), dup[:5])
whole = {}
for r in recs:
    whole.setdefault((r["t"].size, round(float(r["t"].sum()), 3)), []).append(r["file"])
print("Recordings with identical spike counts and sums:", [v for v in whole.values() if len(v) > 1])
pd10 = {}
for q in D["p10"]:
    pd10.setdefault((q["t"].size, round(float(q["t"].sum()), 3)), []).append(q["file"])
print("P10 recordings deposited twice:", [v for v in pd10.values() if len(v) > 1])

In [ ]:
# Is the P10 nerve on the clock of the population? Cross-correlation of 1 s binned rates after the
# stimulus, against the same with the P10 series shifted by a random amount (circularly).
rng = np.random.default_rng(11)
rows = []
for q in D["p10"]:
    r = next(r for r in recs if r["prep"] == q["prep"] and r["bout"] == q["bout"])
    edges = np.arange(33, 122, 0.5)
    pop = np.histogram(r["t"], edges)[0].astype(float)
    nerve = np.histogram(q["t"], edges)[0].astype(float)
    lags = np.arange(-10, 11)
    cc = [np.corrcoef(np.roll(pop, l), nerve)[0, 1] for l in lags]
    shuffled = [np.corrcoef(np.roll(pop, int(rng.integers(20, len(pop) - 20))), nerve)[0, 1] for _ in range(200)]
    rows.append({"file": q["file"], "peak_lag_s": lags[int(np.nanargmax(np.abs(cc)))] * 0.5,
                 "r_at_lag_0": round(cc[10], 3), "max_abs_r": round(np.nanmax(np.abs(cc)), 3),
                 "shifted_95pct_abs_r": round(np.nanpercentile(np.abs(shuffled), 95), 3)})
print("Population rate against P10 rate, 0.5 s bins from 33 s, lags of -5 to 5 s:")
print(pd.DataFrame(rows).to_string(index=False))
# Are neuron ids the same neurons across bouts? Per-neuron rates after onset, correlated between
# bouts of a preparation, against the same with the ids permuted.
rows = []
for p in range(1, 11):
    rs = [np.array([np.sum(tr >= 30) for tr in apl.neuron_trains(r)]) for r in recs if r["prep"] == p]
    for a, b in apl.PAIRS:
        rho = stats.spearmanr(rs[a], rs[b])[0]
        null = [stats.spearmanr(rs[a], rng.permutation(rs[b]))[0] for _ in range(200)]
        rows.append({"prep": p, "pair": f"{a + 1},{b + 1}", "spearman": round(rho, 3),
                     "permuted_95pct": round(np.percentile(null, 95), 3)})
idt = pd.DataFrame(rows)
print("\nRank correlation of neurons' post-onset spike counts between bouts (same id):")
print(idt.pivot(index="prep", columns="pair", values="spearman").to_string())
print("Pairs above the 95th percentile of permuted ids:", int((idt.spearman > idt.permuted_95pct).sum()), "of", len(idt))

In [ ]:
# Coordinates: units and ranges, against the 501 x 501 diode template (plot_maps.m plots x*10, y*10)
for c, p in zip(D["coords"], preps):
    print(f"prep {p['prep']:2d} {c['file']:14s} {c['ganglion']:24s} n={c['x'].size:3d} x {np.nanmin(c['x']):5.1f} to {np.nanmax(c['x']):5.1f}  "
          f"y {np.nanmin(c['y']):5.1f} to {np.nanmax(c['y']):5.1f}  missing {int(np.isnan(c['x']).sum())}")
tpl = D["template"]
ys, xs = np.nonzero(tpl)
print("Diode template", tpl.shape, "nonzero x", xs.min(), "to", xs.max(), "y", ys.min(), "to", ys.max())

**Checks of the description against the data.** Consistent with the description: 30
recordings, three per preparation; 75 to 160 neurons per preparation, 1131 in all; the stimulus at
30 to 32.5 s (in every recording the population rate during the stimulus exceeds the rate before it, and
the rate after 32.5 s exceeds the rate before 30 s in 30 of 30); recordings end at
122.7 s; P10 spikes in nine recordings of preparations 2, 4 and 5 (FileTable lists bouts 1 to 3 for
those three). The P10 intervals are multiples of 0.614 ms, a sampling rate of 1629.7 Hz, which
agrees with the description's 1630 Hz and not with the paper's 1600 Hz. The P10 files of
preparation 2 are named `Jun0414`, its spike file `Jun0414B`; they are matched on the first seven
characters, as the code does. No neuron spike train and no recording appears twice.

The P10 rate and the population rate are correlated after the stimulus at lags of 0 to 5 s in every
P10 recording; in 7 of 9 the largest correlation exceeds the 95th percentile of circularly shifted
series. This is consistent with a shared clock; the lag of a few seconds is expected for a nerve
that fires in one phase of a 10 s cycle, so it does not show the clocks agree to better than a few
seconds. In Jun1114da01 and Jun1114da02 the check does not separate the data from shifted series.

The description says the recordings were concatenated before spike sorting, so that a neuron id
names the same neuron in all three bouts. Consistent with this, per-neuron spike counts after onset
correlate between bouts of a preparation (Spearman 0.63 to 0.96) above the 95th percentile of
permuted ids in 30 of 30 pairs. This is taken as an assumption for the participation claims, which
compare neurons across bouts; their results are given per preparation and bout.

Coordinates hold one (x, y) per neuron, on a 0 to 50 scale (plot_maps.m multiplies by 10 to plot
them on the 501 by 501 diode template). Preparations 7 and 8 are marked unreliable, as the
description says, and preparation 7 has 5 missing positions. No unit is given, and nothing in the
files or the documentation says which direction of x or y is rostral, caudal, medial or lateral.

## dataset-size

> Three bouts of fictive locomotion were evoked in each of 10 isolated preparations (30 population
> responses); each recorded population held 120 to 180 neurons, 1131 neurons in all.

In [ ]:
n_preps = info.prep.nunique()
n_recs = len(info)
per_prep = info.groupby("prep").neurons.first()
print("preparations", n_preps, "| recordings", n_recs, "| neurons over preparations", int(per_prep.sum()))
print("neurons per recording: min", info.neurons.min(), "max", info.neurons.max())
print("recordings with 108 to 198 neurons:", int(info.neurons.between(108, 198).sum()), "of", n_recs)
print("variant, neurons spiking after onset: min", info.neurons_spiking_after_30s.min(), "max",
      info.neurons_spiking_after_30s.max(), "total over preparations",
      int(info.groupby("prep").neurons_spiking_after_30s.first().sum()))
print("per preparation:", per_prep.to_dict())

**Result: partially reproduced.** The counts match: 10 preparations, 30 recordings and 1131
neurons. The range does not: recordings hold 75 to 160 neurons, and only 12 of 30 lie in 108 to 198
(the paper's 120 to 180 widened by 10%). Counting only neurons that spike after onset changes
nothing (75 to 160, 1131). The paper's range cannot hold with its own total: ten preparations of at
least 120 neurons would hold at least 1200.

## fig1d-slow-oscillation

> After stimulation the spike-train power spectra of the populations are dominated by a slow
> oscillation at or below 0.1 Hz.

Chronux `mtspectrumpt` (ported in `apl.mtspectrumpt`, default tapers [3 5], 1 ms, fpass 0 to 1 Hz)
of each neuron's spikes after onset, averaged over the neurons of a recording.

In [ ]:
t0 = time.time()
spectra = []
for r in recs:
    trains = [tr[tr >= apl.STIM_ON] for tr in apl.neuron_trains(r)]
    f, S = apl.mtspectrumpt(trains)
    spectra.append(S)
spectra = np.array(spectra)
grand = spectra.mean(0)
fpk = np.array([apl.peak_frequency(f, S) for S in spectra])
fmax = np.array([f[1:][np.argmax(S[1:])] for S in spectra])
print(f"spectra in {time.time() - t0:.0f} s; frequency grid {f[0]:.4f} to {f[-1]:.4f} Hz, step {f[1] - f[0]:.4f} Hz")
print("grand mean spectrum: weighted peak", round(apl.peak_frequency(f, grand), 4), "Hz; maximum above 0 Hz at",
      round(f[1:][np.argmax(grand[1:])], 4), "Hz")
print("per recording, weighted peak (code): min", fpk.min().round(4), "median", np.median(fpk).round(4), "max", fpk.max().round(4))
print("recordings with weighted peak above 0 and at or below 0.11 Hz:", int(((fpk > 0) & (fpk <= 0.11)).sum()), "of 30")
print("variant, frequency of maximum power above 0 Hz: min", fmax.min().round(4), "median", np.median(fmax).round(4),
      "max", fmax.max().round(4), "; at or below 0.11 Hz:", int((fmax <= 0.11).sum()), "of 30")
print(pd.DataFrame({"file": [r["file"] for r in recs], "weighted_peak_Hz": fpk.round(4), "max_power_Hz": fmax.round(4)}).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3))
for S in spectra:
    ax.plot(f, S, color="0.7", lw=0.5)
ax.plot(f, grand, "k")
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("power"); ax.set_title("post-onset spectra (all points shown)")
plt.show()

In [ ]:
# Variant (not judged): is the low-frequency peak the slow decay of the rate (claim fig1e) or an
# oscillation? The population rate from stimulation offset in 0.1 s bins, minus a quadratic fit in
# time (the decay), with a multitaper spectrum (DPSS, time-bandwidth 3, 5 tapers, as the main
# spectrum). Reported: the frequency of maximum power above 0.02 Hz, the code's weighted peak, and
# the power in 0.05 to 0.11 Hz relative to 0.2 to 1 Hz.
from scipy.signal.windows import dpss
rows = []
for r in recs:
    edges = np.arange(apl.STIM_OFF, r["t"].max(), 0.1)
    y = np.histogram(r["t"], edges)[0].astype(float) / 0.1
    tt = edges[:-1] + 0.05
    y_d = y - np.polyval(np.polyfit(tt, y, 2), tt)
    tp = dpss(y.size, 3, 5)
    nfft = int(2 ** np.ceil(np.log2(y.size)) * 2)
    fr = np.fft.rfftfreq(nfft, 0.1)
    Sd = np.mean(np.abs(np.fft.rfft(tp * y_d, nfft)) ** 2, 0)
    keep = (fr > 0) & (fr <= 1)
    fk, Sk = fr[keep], Sd[keep]
    band = Sk[(fk >= 0.05) & (fk <= 0.11)].mean() / Sk[(fk >= 0.2) & (fk <= 1)].mean()
    above = fk > 0.02
    rows.append({"file": r["file"], "fmax_detrended": fk[above][np.argmax(Sk[above])], "weighted_peak_detrended": apl.peak_frequency(fk, Sk),
                 "band_ratio_0.05-0.11_vs_0.2-1": band})
DT1 = pd.DataFrame(rows)
print(f"frequency grid step {fk[1] - fk[0]:.4f} Hz")
print("detrended population rate, frequency of maximum power above 0.02 Hz: min", round(DT1.fmax_detrended.min(), 4), "median",
      round(DT1.fmax_detrended.median(), 4), "max", round(DT1.fmax_detrended.max(), 4), "; at or below 0.11 Hz:",
      int((DT1.fmax_detrended <= 0.11).sum()), "of 30")
print("weighted peak of the detrended spectrum: min", round(DT1.weighted_peak_detrended.min(), 4), "median",
      round(DT1.weighted_peak_detrended.median(), 4), "max", round(DT1.weighted_peak_detrended.max(), 4), "; at or below 0.11 Hz:",
      int((DT1.weighted_peak_detrended <= 0.11).sum()), "of 30")
print("mean power 0.05 to 0.11 Hz over mean power 0.2 to 1 Hz: min", round(DT1["band_ratio_0.05-0.11_vs_0.2-1"].min(), 1), "median",
      round(DT1["band_ratio_0.05-0.11_vs_0.2-1"].median(), 1), "; above 1 in", int((DT1["band_ratio_0.05-0.11_vs_0.2-1"] > 1).sum()), "of 30")
print("grand mean point-process spectrum at its first bins (Hz: power):", {round(float(a), 4): round(float(b), 3) for a, b in zip(f[1:8], grand[1:8])})

**Result: reproduced.** The grand mean spectrum's power-weighted peak is at 0.0486 Hz, and
the peaks of all 30 recordings lie above 0 and at or below 0.11 Hz (0.0131 to 0.0789 Hz). The
variant, the frequency of maximum power above 0 Hz, lies at 0.0076 to 0.0992 Hz, also 30 of 30 at or
below 0.11 Hz. A detrended check, not judged, tells the oscillation from the decay
of the rate (claim fig1e): with a quadratic trend removed from the population rate after offset, the
frequency of maximum power above 0.02 Hz is 0.034 to 0.156 Hz (median 0.064 Hz), at or below 0.11 Hz
in 27 of 30, and the mean power in 0.05 to 0.11 Hz exceeds that in 0.2 to 1 Hz in 30 of 30 (median
ratio 9.8). The grand mean point-process spectrum is nearly flat over its first bins (7.54 at
0.0076 Hz, 7.24 at 0.053 Hz), so its maximum at the lowest bin reflects the decay as well as the
oscillation.

## fig1e-decaying-rate

> The population firing rate decays slowly after the stimulus, decreasing in most evoked responses.

In [ ]:
rows = []
for r in recs:
    t = r["t"]
    starts = np.arange(30, r["t"].max() - 20 + 1e-9, 5)
    counts = np.array([np.sum((t >= s) & (t < s + 20)) for s in starts])
    R = np.corrcoef(starts + 10, counts)[0, 1]
    edges = np.arange(32.5, r["t"].max(), 1.0)
    rate = np.histogram(t, edges)[0]
    slope = np.polyfit(edges[:-1] + 0.5, rate, 1)[0]
    rows.append({"prep": r["prep"], "bout": r["bout"], "R_20s_windows": R, "n_windows": len(starts), "slope_spikes_per_s2": slope})
rate_tab = pd.DataFrame(rows)
print(rate_tab.round(3).to_string(index=False))
print("(a) decreasing, R < -0.2:", int((rate_tab.R_20s_windows < -0.2).sum()), "of 30; increasing, R > 0.2:",
      int((rate_tab.R_20s_windows > 0.2).sum()))
print("(b) negative slope from 32.5 s:", int((rate_tab.slope_spikes_per_s2 < 0).sum()), "of 30")
print("per preparation, (a) decreasing:", rate_tab.assign(dec=rate_tab.R_20s_windows < -0.2).groupby("prep").dec.sum().to_dict())

**Result: reproduced.** Under reading (a), all 30 responses are decreasing (Pearson R of
window time against count from -0.992 to -0.774, all below -0.2) and none is increasing; under
reading (b) all 30 slopes are negative. Every preparation has three decreasing responses.

## fig2b-dimensions

> Capturing 80% of the population variance needs 4 to 8 embedding dimensions in the evoked
> responses, a reduction by more than a factor of 10 from the number of neurons.

The Gaussian SD is read four ways (helper docstring) and the count two ways, as the criterion
lists them.

In [ ]:
sds = apl.gaussian_sds(recs)
print("Gaussian SD (s) by reading: ", {k: (round(v.min(), 3), round(v.max(), 3)) for k, v in sds.items()})
dims = {}
t0 = time.time()
for mode in ["sqrt12", "twelve", "own", "pooled"]:
    evs = [apl.Program(r, s).ev for r, s in zip(recs, sds[mode])]
    for rule in ["min80", "le80"]:
        dims[(mode, rule)] = np.array([apl.n_pcs(ev, 0.8, rule) for ev in evs])
print(f"PCA under every reading in {time.time() - t0:.0f} s")
nn = info.neurons.values
rows = []
for (mode, rule), d in dims.items():
    rows.append({"sd": mode, "count": rule, "min": d.min(), "max": d.max(), "in_3.6_to_8.8": int(((d >= 3.6) & (d <= 8.8)).sum()),
                 "min_neurons_over_dims": round((nn / d).min(), 1), "ratio_over_10": int((nn / d > 10).sum())})
dim_tab = pd.DataFrame(rows)
print(dim_tab.to_string(index=False))
print("\ncounts per recording (min80 | le80) for the code's width (own median ISI):")
print(pd.DataFrame({"file": info.file, "own_min80": dims[("own", "min80")], "own_le80": dims[("own", "le80")],
                    "pooled_min80": dims[("pooled", "min80")], "pooled_le80": dims[("pooled", "le80")], "neurons": nn}).to_string(index=False))

**Result: reproduced, under the readings with the code's width and the code's count.** The criterion asks for one reading. With
the median ISI as the SD and the count of PCs whose cumulative variance is at most 80% (own/le80),
28 of 30 counts lie in 3.6 to 8.8 (3 to 7) and the number of neurons divided by the count exceeds
10 in all 30 (lowest 10.7); pooled/le80 gives the same. With the Methods' count (smallest number
reaching 80%) the counts are 4 to 8 for own/min80, the paper's range, all 30 in 3.6 to 8.8, but the
ratio exceeds 10 in 29 of 30 (lowest 9.4, 75 neurons and 8 PCs). The Methods' printed width gives 21 to 76
dimensions (median ISI / sqrt(12), 21 to 41; median ISI / 12, 33 to 76) and ratios of 1.8 to 2.8 at the lowest; it
cannot give the paper's counts. The adopted reading of the later claims (own/min80) is therefore one
under which this claim fails, by one response; its counts are the paper's 4 to 8.

## The recurrence analysis

Run once per reading of the width and the count. The local linear models (claims fig4 to fig5)
are fitted for the four readings with the code's width; the two narrow widths give 21 to 76
dimensions (claim fig2b-dimensions) and are carried only through the dominant period
(fig2ef-periodic-recurrence). The adopted reading, used where one reading has to be chosen, is
`own/min80`: the median ISI of the recording (the width the code uses; the Methods compute it
"in population", the recording) and the smallest number of PCs reaching 80% (the Methods, "at least
80% of the variance"). The other readings are shown with it.

No single reading makes every claim that depends on it hold. Under the adopted reading claim
fig2b fails by one response (75 neurons over 8 PCs, a ratio of 9.4), though its counts are the
paper's 4 to 8; own/le80 passes fig2b, fig3h and fig6s2 but gives a linear-model period of 13.55 s
for fig4c. The Summary tabulates the outcome under each reading for the claims that change.

In [ ]:
t0 = time.time()
RUNS = {}
for mode in ["own", "pooled"]:
    for rule in ["min80", "le80"]:
        RUNS[f"{mode}/{rule}"] = apl.run_all(recs, sds[mode], rule, lld=True, workers=4)
for mode in ["sqrt12", "twelve"]:
    for rule in ["min80", "le80"]:
        RUNS[f"{mode}/{rule}"] = apl.run_all(recs, sds[mode], rule, lld=False, workers=4)
SUM = {k: pd.DataFrame([apl.summarise(o) for o in v]) for k, v in RUNS.items()}
print(f"recurrence analysis of 30 recordings under {len(RUNS)} readings in {time.time() - t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6:.2f} GB per worker")
ADOPT = "own/min80"
print("checked points per recording:", SUM[ADOPT].n_points.min(), "to", SUM[ADOPT].n_points.max(), "(Methods: around 7770)")

## fig2ef-periodic-recurrence

> The evoked low-dimensional activity recurs with a regular period; strongly periodic recurrence on
> the scale of 10 to 15 s is found in many but not all of the 30 responses.

In [ ]:
rows = []
for k, S in SUM.items():
    for stat in ["period_mean", "period_median"]:
        v = S[stat].values
        rows.append({"reading": k, "period": stat, "n_with_peak": int(np.isfinite(v).sum()),
                     "in_9_to_16.5": int(((v >= 9) & (v <= 16.5)).sum()), "min": np.nanmin(v).round(2),
                     "median": np.nanmedian(v).round(2), "max": np.nanmax(v).round(2)})
per_tab = pd.DataFrame(rows)
per_tab["reproduced"] = (per_tab["in_9_to_16.5"] > 15) & (per_tab["in_9_to_16.5"] < 30)
print(per_tab.to_string(index=False))
S = SUM[ADOPT]
print("\nadopted reading, per recording:")
print(S[["prep", "bout", "d", "n_peaks", "prop_recurrent", "period_mean", "period_median"]].round(3).to_string(index=False))
print("by preparation, mean period:", S.groupby("prep").period_mean.mean().round(2).to_dict())
print("adopted reading, quartiles of the dominant period (mean of the peak):", np.percentile(S.period_mean, [25, 75]).round(2),
      "; outside 9 to 16.5 s:", S[~S.period_mean.between(9, 16.5)][["prep", "bout", "period_mean"]].round(2).values.tolist())

**Result: reproduced.** Under the adopted reading, 26 of 30 dominant periods (mean of the
peak) lie in 9 to 16.5 s, and 4 do not (17.28, 22.23, 8.75 and 19.94 s); the quartiles are 11.06
and 14.98 s. With the median of the peak 23 of 30 do. All four readings with the code's width give
22 to 26 of 30, and the readings with the narrow widths 13 to 29; every reading but twelve/le80
with the mean (13 of 30) meets the criterion.

## fig2e-single-dominant-orbit

> Almost all populations are dominated by a single periodic orbit; the majority of programs have
> more than 50% of their recurrent points in the dominant peak.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    S = SUM[k]
    rows.append({"reading": k, "over_50pct_hist": int((S.prop_dom_hist > 0.5).sum()),
                 "over_50pct_all_recurrent": int((S.prop_dom_all > 0.5).sum()),
                 "peaks_min": S.n_peaks.min(), "peaks_max": S.n_peaks.max(),
                 "peaks_counts": S.n_peaks.value_counts().sort_index().to_dict()})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading:")
print(SUM[ADOPT][["prep", "bout", "n_peaks", "dom_n", "prop_dom_hist", "prop_dom_all"]].round(3).to_string(index=False))

**Result: reproduced.** Under the adopted reading, 29 of 30 responses have more than 50% of
their recurrent points (delays of 5 to 50 s) in the dominant peak, and 27 of 30 counting all
recurrent points (the code's propInMaxPeriod). Responses have 1 to 4 peaks, as the Methods say;
12 have one. The other readings give 29 to 30 and 24 to 27.

## fig2s2-threshold-robust

> The periodic orbits are robust to the threshold for recurrence, with similar periodicity at
> thresholds of 10%, 5% and 2%.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    P = {th: pd.DataFrame([apl.summarise(o, i) for o in RUNS[k]]) for i, th in enumerate(apl.THETA_PRCT)}
    for stat in ["period_mean", "period_median"]:
        p10 = P[10][stat].values
        for th in (5, 2):
            pl = P[th][stat].values
            ok = np.isfinite(p10) & np.isfinite(pl)
            rho, p = stats.spearmanr(p10[ok], pl[ok])
            mad = np.median(np.abs(pl[ok] - p10[ok]))
            rows.append({"reading": k, "period": stat, "theta": f"{th}%", "n_both": int(ok.sum()),
                         "spearman": round(rho, 3), "p": p, "(a)": bool(rho > 0 and p < 0.05),
                         "median_abs_diff_s": round(mad, 2), "10pct_of_median": round(0.1 * np.median(p10[ok]), 2),
                         "(b)": bool(mad < 0.1 * np.median(p10[ok]))})
thr_tab = pd.DataFrame(rows)
print(thr_tab.to_string(index=False))
P = {th: pd.DataFrame([apl.summarise(o, i) for o in RUNS[ADOPT]]) for i, th in enumerate(apl.THETA_PRCT)}
print("\nadopted reading, mean dominant period at each threshold:")
print(pd.DataFrame({th: P[th].period_mean.round(2) for th in P}).assign(file=info.file).to_string(index=False))

**Result: reproduced.** Under every reading, the dominant periods at the 5% and 2%
thresholds correlate with those at 10% (Spearman 0.70 to 0.97, all p < 2e-5), so reading (a) holds
for both lower thresholds. Reading (b) holds at 5% in every case and fails at 2% in six of eight
cases (median absolute difference 1.30 s against 1.27 s for the adopted reading with the mean).

## fig3a-coalescence

> Almost all evoked responses reach a state of high recurrence within one oscillation period of
> stimulation and are thereafter dominated by recurrence.

The coalescence time is the mid-point of the first 5 s window with at least 90% recurrent points (Methods).
Windows start at 35 s, where the checked points start (Methods), so the earliest possible value is
37.5 s, 7.5 s after onset.

In [ ]:
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    S = SUM[k]
    c_on = S.coalesce_t - apl.STIM_ON
    for stat in ["period_mean", "period_median"]:
        rows.append({"reading": k, "period": stat, "within_period_from_onset": int((c_on <= S[stat]).sum()),
                     "within_period_from_offset": int((c_on - 2.5 <= S[stat]).sum()),
                     "recurrent_after_over_50pct": int((S.prop_recurrent_after > 0.5).sum()),
                     "never_coalesced": int(S.coalesce_t.isna().sum())})
print(pd.DataFrame(rows).to_string(index=False))
S = SUM[ADOPT].assign(coal_from_onset=lambda d: d.coalesce_t - 30)
print("\nadopted reading: coalescence from onset, s: min", S.coal_from_onset.min(), "median", S.coal_from_onset.median(),
      "max", S.coal_from_onset.max(), "; proportion recurrent after it: min", S.prop_recurrent_after.min().round(3),
      "median", S.prop_recurrent_after.median().round(3))
print(S[["prep", "bout", "coal_from_onset", "period_mean", "period_median", "prop_recurrent_after"]].round(3).to_string(index=False))

In [ ]:
# Why coalescence is late: recurrence density of the first windows (window starts from 35 s, so the
# earliest mid-point is 7.5 s after onset), adopted reading.
rows = []
for o, s in zip(RUNS[ADOPT], SUM[ADOPT].itertuples()):
    w = o["windows"][-1]
    within = w["mid_t"] - apl.STIM_ON <= s.period_mean
    rows.append({"prep": o["prep"], "bout": o["bout"], "first_window_density": w["dens"][0],
                 "max_density_within_one_period": w["dens"][within].max() if within.any() else np.nan,
                 "windows_within_one_period": int(within.sum()), "coalescence_from_onset": s.coalesce_t - apl.STIM_ON,
                 "period_mean": s.period_mean})
CO = pd.DataFrame(rows)
print(CO.round(3).to_string(index=False))
print("responses whose highest density within one period of onset is at least 0.8:", int((CO.max_density_within_one_period >= 0.8).sum()),
      "; at least 0.7:", int((CO.max_density_within_one_period >= 0.7).sum()), "of 30")
fig, ax = plt.subplots(figsize=(9, 3))
for o in RUNS[ADOPT][:6]:
    w = o["windows"][-1]
    ax.plot(w["mid_t"] - apl.STIM_ON, w["dens"], lw=1, label=f"prep {o['prep']} bout {o['bout']}")
ax.axhline(0.9, color="k", lw=0.5); ax.set_xlabel("window mid-point after onset (s)"); ax.set_ylabel("recurrence density")
ax.set_ylim(0, 1.02); ax.legend(fontsize=7, ncol=3); plt.tight_layout(); plt.show()
print("plotted: the first six responses, densities 0 to 1")

**Result: not reproduced.** Under the adopted reading, the coalescence time (first window with at
least 90% recurrent points) is within one dominant period of stimulation onset in 12 of 30
responses (19 of 30 measured from offset), where the criterion asks for 27; the other readings give
11 to 14 (17 to 20 from offset). Coalescence comes 7.5 to 49.5 s after onset (median 14.5 s). The
second part holds: after coalescence more than 50% of the points recur in 30 of 30 responses (60.8%
to 98.6%, median 86%). Because the checked points start at 35 s, the earliest possible coalescence
is 7.5 s after onset, close to the shortest periods (8.7 to 9.4 s). The late coalescence is not an
edge effect of the 90% line: within one period of onset the highest window density reaches 0.8 in
15 of 30 responses and 0.7 in 21.

## Comparisons between the programs of a preparation

Hausdorff distances on the axes of program 1 (claim fig3c) and on common axes (fig3s1a), with 100
neuron shuffles each way as control; the distances of Figure 3D; the similarity matrices of
Figure 3F. Run for the four readings with the code's width (the point sets come from each
reading's recurrence analysis).

In [ ]:
t0 = time.time()
MAN = {k: apl.run_manifold(recs, sds[k.split("/")[0]], RUNS[k], n_shuffle=100, seed=1)
       for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]}
print(f"comparisons in {time.time() - t0:.0f} s")
def haus_table(k, axes, ps):
    rows = []
    for m in MAN[k]:
        for (i, j) in apl.PAIRS:
            h, c = m[(axes, ps, i, j)]
            rows.append({"prep": m["prep"], "pair": f"{i + 1},{j + 1}", "H": h, "ctrl_mean": np.mean(c),
                         "ctrl_2sem": 2 * np.std(c, ddof=1) / np.sqrt(c.size), "dims": m["k_prog1" if axes == "prog1" else "k_common"]})
    return pd.DataFrame(rows)

## fig3c-same-manifold

> Sequentially evoked responses of a preparation fall on the same manifold; their distance is below
> that of random projections for all pairs except two, pairs (1,2) and (1,3) of preparation 4.

In [ ]:
rows = []
for k in MAN:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "prog1", ps)
        below = T.H < T.ctrl_mean
        rows.append({"reading": k, "points": ps, "pairs_below_control": int(below.sum()),
                     "failing_pairs": [(p, q) for p, q in zip(T.prep[~below], T.pair[~below])]})
print(pd.DataFrame(rows).to_string(index=False))
T = haus_table(ADOPT, "prog1", "recurrent")
print("\nadopted reading, recurrent points:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))
T = haus_table(ADOPT, "prog1", "code")
print("\nadopted reading, the code's point sets:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))

**Result: reproduced.** With the code's point sets, 28 of 30 pairs are closer than their
random-projection control under every reading, and the two that fail are pairs (1,2) and (1,3) of
preparation 4, as the paper names them (distance 1.19 and 1.17 times the control under the adopted
reading). With the recurrent points (the Methods), 29 of 30 pass under every reading and only pair
(1,2) of preparation 4 fails (1.04 times the control). The control's 2 s.e.m. is 0.11 to 0.36,
small against the differences.

## fig3s1a-common-axes

> The convergence of sequential responses to the same manifold holds when the three programs are
> projected onto common axes from a PCA of their concatenated activity.

In [ ]:
rows = []
for k in MAN:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "common", ps)
        below = T.H < T.ctrl_mean
        rows.append({"reading": k, "points": ps, "pairs_below_control": int(below.sum()),
                     "failing_pairs": [(p, q) for p, q in zip(T.prep[~below], T.pair[~below])],
                     "dims": sorted(set(T.dims))})
print(pd.DataFrame(rows).to_string(index=False))
T = haus_table(ADOPT, "common", "code")
print("\nadopted reading, the code's point sets:")
print(T.assign(ratio=T.H / T.ctrl_mean).round(3).to_string(index=False))

**Result: reproduced.** On common axes (5 to 10 PCs), 28 of 30 pairs pass with the code's
point sets (failing: preparation 4, pairs (1,2) and (1,3)) under three readings, and 27 under
own/le80 (also preparation 8, pair (2,3), at 0.99 of its control in the adopted reading's table).
With the recurrent points all 30 pass under every reading.

## fig3d-initial-conditions

> Stimulation drives responses of the same preparation far apart before coalescence, more than at
> stimulus onset or after both have coalesced, so the same manifold is reached from different
> initial conditions.

The transient distance is the largest distance between a point of one program and a point of the
other between onset and their coalescence; for speed every fifth point of the second program is
used (a standard choice: neighbouring points 10 ms apart are nearly equal).

In [ ]:
rows = []
for k in MAN:
    for ps in ["code", "recurrent"]:
        for m in MAN[k]:
            for (i, j) in apl.PAIRS:
                on, tr = m[("ic", i, j)]
                rows.append({"reading": k, "points": ps, "prep": m["prep"], "pair": f"{i + 1},{j + 1}",
                             "onset": on, "transient": tr, "coalesced": m[("common", ps, i, j)][0]})
IC = pd.DataFrame(rows)
summ = IC.groupby(["reading", "points"]).apply(lambda d: pd.Series({
    "transient_over_both": int(((d.transient > d.onset) & (d.transient > d.coalesced)).sum()),
    "median_onset": d.onset.median(), "median_transient": d.transient.median(), "median_coalesced": d.coalesced.median()}))
print(summ.round(2).to_string())
print("\nadopted reading, code's point sets:")
print(IC[(IC.reading == ADOPT) & (IC.points == "code")].drop(columns=["reading", "points"]).round(2).to_string(index=False))

**Result: reproduced.** In 30 of 30 pairs, under every reading and both point sets, the
largest distance between onset and coalescence exceeds both the distance at onset and the
coalesced Hausdorff distance. Under the adopted reading the medians are 9.31 at onset, 45.44 in the
transient and 14.51 after coalescence (code's point sets).

## fig3f-pairwise-similarity

> Pairwise similarity between neurons is preserved across responses of the same preparation, above a
> null model, except for pairs (1,3) and (2,3) of preparation 7.

In [ ]:
rows, detail = [], []
for k in MAN:
    for meth in ["spearman", "pearson"]:
        corr = (lambda a, b: stats.spearmanr(a, b)[0]) if meth == "spearman" else (lambda a, b: np.corrcoef(a, b)[0, 1])
        for ctl in ["first", "larger"]:
            n, fails = 0, []
            for m in MAN[k]:
                Sm, Em = m["sim"]
                cs = [corr(Sm[i].ravel(), Em[i].ravel()) for i in range(3)]
                for (i, j) in apl.PAIRS:
                    dat = corr(Sm[i].ravel(), Sm[j].ravel())
                    c = cs[i] if ctl == "first" else max(cs[i], cs[j])
                    if dat > c:
                        n += 1
                    else:
                        fails.append((m["prep"], f"{i + 1},{j + 1}"))
                    if k == ADOPT and ctl == "first":
                        detail.append({"corr": meth, "prep": m["prep"], "pair": f"{i + 1},{j + 1}", "data": dat,
                                       "control_first": cs[i], "control_second": cs[j]})
            rows.append({"reading": k, "corr": meth, "control": ctl, "pairs_above": n, "failing": fails})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading:")
print(pd.DataFrame(detail).round(3).to_string(index=False))

**Result: reproduced.** Under every reading of width and count, the data correlation exceeds
the control in 29 of 30 pairs (Spearman, control of the first program), 30 (Pearson, first), 27
(Spearman, larger control) or 28 (Pearson, larger control); all are at least 26. With Pearson
correlation and the larger control the two failing pairs are (1,3) and (2,3) of preparation 7, as
the paper names them. With the first program's control the Spearman reading fails pair (1,3) of
preparation 4 instead (0.597 against 0.622).

## fig3h-perturbations-return

> Of 19 spontaneous divergences from the trajectory, 17 (90%) returned to a recurrent state; the two
> that did not were consistent with the end of the evoked locomotion.

A divergent period (Methods) is a run of windows after coalescence with recurrence density below
90% that falls below 50%; "returned" means a later window is above 90%. Periods whose divergent
point lies within two dominant periods of the end are removed (Methods), with the dominant period
read as the mean or the median of the peak. The code removes only periods that never return
(`Are_jumps_perturbations_or_transitions2.m`); that is shown as a variant.

In [ ]:
DIV = {}
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    for stat in ["period_mean", "period_median"]:
        allv = []
        for o, s in zip(RUNS[k], SUM[k].itertuples()):
            for d in apl.divergences(o, getattr(s, stat)):
                d.update(prep=o["prep"], bout=o["bout"])
                allv.append(d)
        V = pd.DataFrame(allv)
        DIV[(k, stat)] = V
        kept = V[~V.near_end]
        code_kept = V[V.returned | ~V.near_end]
        rows.append({"reading": k, "period": stat, "all_periods": len(V), "kept": len(kept),
                     "returned": int(kept.returned.sum()), "fraction": round(kept.returned.mean(), 3),
                     "code_rule_kept": len(code_kept), "code_rule_returned": int(code_kept.returned.sum())})
print(pd.DataFrame(rows).to_string(index=False))
V = DIV[(ADOPT, "period_mean")]
print("\nadopted reading, mean period:")
print(V[["prep", "bout", "t_point", "min_dens", "returned", "near_end"]].round(3).to_string(index=False))

**Result: partially reproduced, resting on the adopted reading.** Under the adopted reading
with the Methods' period (mean of the dominant peak), 34 divergent periods are found and 16 remain
after removing those within two periods of the end; 15 of 16 (0.938) returned above 90%. The fraction
matches (at least 80%), but both counts fall just short of their tolerances: 16 divergent periods
against 17.1 to 20.9, and 15 returns against 15.3 to 18.7. With the median period (the code's) 17 remain and 16
return (0.941); 16 returns are within tolerance, but 17 periods are just below 17.1. Over the four readings 14 to 19 periods remain and 13 to 17
return, 0.875 to 0.941. The code's rule (dropping only the periods within two cycles of the end that never return) keeps
17 to 21 over the readings, and under the adopted reading it keeps 19 periods of which 18 return
(0.947), with either period, which meets both tolerances and matches the paper's 19. The criterion
states the Methods' rule ("We removed all such divergent periods whose divergent point fell within
two oscillation cycles of the end of the recording", Testing return to the same manifold after
perturbation), so the code's rule is reported, not judged. Under the Methods' rule the counts meet
the criterion only under own/le80, with either period (18 kept and 16 returned; 19 and 17). With
eight combinations run, a pass under one of them is weak evidence, and the adopted reading's result
decides. The criterion has no partial outcome; it is recorded as partially reproduced because
the returned fraction holds and the two counts miss their tolerances by 1.1 and 0.3.

## fig3i-same-manifold-after

> Of the 17 divergences that returned to a recurrent state, all but three returned to the same
> manifold; the three appeared to move to a different periodic attractor.

In [ ]:
rows = []
for (k, stat), V in DIV.items():
    kept = V[~V.near_end & V.returned]
    rows.append({"reading": k, "period": stat, "returned": len(kept),
                 "same_(a)_after_end": int((kept.p_after_end > 0.2).sum()),
                 "same_(b)_after_point": int((kept.p_after_point > 0.2).sum())})
print(pd.DataFrame(rows).to_string(index=False))
V = DIV[(ADOPT, "period_mean")]
print("\nadopted reading, returned periods:")
print(V[~V.near_end & V.returned][["prep", "bout", "t_point", "t_return", "n_before", "p_after_end", "p_after_point"]].round(3).to_string(index=False))

**Result: reproduced under reading (b).** Of the 15 returned divergences of the adopted
reading, 13 have more than 20% of the recurrences of the final window before them falling after the
divergent point (reading (b), the code's blnSame), within 12.6 to 15.4; under reading (a), after the
end of the divergent period, 10 do. With the median period, 14 of 16 under (b) and 11 under (a). Over
the four readings, (b) gives 11 to 17 and (a) 9 to 14. The count depends strongly on the reading (9 to 17), and the
criterion's "under a reading" is met under the adopted reading, (b); this is weak evidence for the
paper's 14.

## fig4bc-stable-spiral

> The evoked responses have highly clustered maximum eigenvalues of the local linear dynamics,
> dominated by negative complex values, so the pedal ganglion implements a contracting periodic
> orbit, a stable spiral attractor, found in every preparation.

In [ ]:
EIG = {}
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    E = pd.DataFrame([{**{x: y for x, y in apl.emax_stats(o["eigs"], o["npts"]).items() if x != "eigs"},
                       "prep": o["prep"], "bout": o["bout"]} for o in RUNS[k]])
    E["spiral"] = (E.a < -0.0001) & (E.b > 0.0005)
    EIG[k] = E
    per_prep = E.groupby("prep").spiral.sum()
    rows.append({"reading": k, "mean_a": E.a.mean(), "mean_b": E.b.mean(), "sd_a": E.a.std(), "sd_b": E.b.std(),
                 "grand_mean_spiral": bool(E.a.mean() < -0.0001 and E.b.mean() > 0.0005),
                 "responses_spiral": int(E.spiral.sum()), "preps_2of3": int((per_prep >= 2).sum()),
                 "used_points_min": E.n_used.min(), "dropped_points": int((E.n_all - E.n_used).sum())})
print(pd.DataFrame(rows).to_string(index=False))
E = EIG[ADOPT]
print("\nadopted reading:")
print(E[["prep", "bout", "a", "b", "sd_a", "sd_b", "n_used", "n_all", "spiral"]].to_string(index=False))
print("spiral responses per preparation:", E.groupby("prep").spiral.sum().to_dict())
npts = [o["npts"] for o in RUNS[ADOPT]]
print("neighbours per point of the dominant orbit: fewest in any response", min(n.min() for n in npts),
      "; median of the per-response medians", np.median([np.median(n) for n in npts]),
      "; per-response 5th percentiles", [int(np.percentile(n, 5)) for n in npts])
fig, ax = plt.subplots(figsize=(4, 3.5))
ax.scatter(E.b, E.a, c=E.prep, cmap="tab10", s=15)
ax.axhline(-0.0001, color="0.6", lw=0.5); ax.axvline(0.0005, color="0.6", lw=0.5)
ax.set_xlabel("mean imaginary part b"); ax.set_ylabel("mean real part a")
ax.set_title(f"all 30 shown; a {E.a.min():.4f} to {E.a.max():.4f}")
plt.show()

**Result: partially reproduced.** The mean over the 30 responses is a stable spiral under
every reading (adopted: a = -0.000362, b = 0.00729), but responses are not all spirals: 21 of 30
are in the stable-spiral region, and 8 of 10 preparations have at least two of three (preparations 3
and 4 have one each). The other readings give 20 to 23 responses and 6 to 9 preparations. The
spread across responses is large against the mean of a (SD 0.00058 against -0.00036; SD of b
0.0028 against 0.0073). No point of the dominant orbits had fewer than 100 neighbours: the fewest in any response is 195
and the median of the per-response medians 1566, so the 100-neighbour filter of the code removes nothing here.

## fig4c-orbital-period

> The linear model estimates a mean orbital period of about 10 s, consistent with the period from
> the recurrent points.

In [ ]:
rows = []
for k, E in EIG.items():
    lin = 2 * np.pi / E.b * 0.01
    S = SUM[k]
    r_mean = stats.pearsonr(lin, S.period_mean)
    r_med = stats.pearsonr(lin, S.period_median)
    rows.append({"reading": k, "mean_linear_period_s": lin.mean(), "variant_mean_of_mRot_s": E.mrot.mean(),
                 "pearson_r_vs_mean_recurrence": r_mean[0], "p": r_mean[1], "pearson_r_vs_median_recurrence": r_med[0], "p_med": r_med[1],
                 "min": lin.min(), "max": lin.max()})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced, resting on the adopted reading.** The mean over the 30 responses of
2 pi / b times 0.01 s is 10.45 s under the adopted reading (9 to 11 s required); the variant, the
mean of the per-point rotation periods, is 10.24 s. Under the code's count rule the means are 13.55
s (own/le80) and 12.22 s (pooled/le80), outside the tolerance. The per-response linear periods do
not correlate with the recurrence periods (Pearson r = -0.063, p = 0.74, adopted; -0.12 to 0.03 for
the other readings), and the means do not agree within the claim's tolerance either: 10.45 s against a median
recurrence period of 12.73 s, 2.3 s apart where the tolerance is 1 s. The criterion judges the mean
against 10 s and reports the correlation, so the statement's "consistent with" is reported, not
judged; neither the correlation nor the means support it.

## fig4s1a-contracting-dimensions

> In most responses the trajectory contracts in all embedding dimensions; a few responses have a
> positive real eigenvalue in one low-variance dimension, a minor expansion.

In [ ]:
rows, detail = [], []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    allneg = one_pos_later = 0
    for o in RUNS[k]:
        st = apl.emax_stats(o["eigs"], o["npts"])
        ed = apl.eig_dimensions(st["eigs"])
        mu = ed["mean_unique"]
        allneg += bool(np.all(mu < 0))
        one_pos_later += bool(np.sum(mu > 0) == 1 and mu[0] < 0)
        if k == ADOPT:
            detail.append({"prep": o["prep"], "bout": o["bout"], "unique_dims": mu.size, "positive": int(np.sum(mu > 0)),
                           "first_positive": bool(mu[0] > 0), "means": np.round(mu * 1000, 3).tolist()})
    rows.append({"reading": k, "all_negative": allneg, "one_positive_not_first": one_pos_later})
print(pd.DataFrame(rows).to_string(index=False))
print("\nadopted reading (means x 1000):")
print(pd.DataFrame(detail).to_string(index=False))

**Result: not reproduced.** Only 11 of 30 responses have negative mean real parts in all
unique dimensions under the adopted reading (8 to 11 under the others), where the criterion asks for
more than 15. The second part holds: 10 responses (9 to 12) have exactly one positive mean, in a
dimension other than the first, which is at least one and fewer than half. In 7 responses the first
dimension's mean is positive.

## fig4s1b-rotation-dominant

> Local trajectories were dominated by rotation throughout; in every preparation most local models
> have a complex eigenvalue among their top two.

In [ ]:
rows, detail = [], []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    pc = []
    for o in RUNS[k]:
        st = apl.emax_stats(o["eigs"], o["npts"])
        ed = apl.eig_dimensions(st["eigs"])
        pc.append({"prep": o["prep"], "bout": o["bout"], "p_complex12": ed["p_complex12"], "p_complex1": ed["p_complex1"]})
    pc = pd.DataFrame(pc)
    maj = pc.assign(o=pc.p_complex12 > 0.5).groupby("prep").o.sum()
    rows.append({"reading": k, "responses_over_0.5": int((pc.p_complex12 > 0.5).sum()), "min": pc.p_complex12.min(),
                 "preps_with_majority": int((maj >= 2).sum())})
    if k == ADOPT:
        detail = pc
print(pd.DataFrame(rows).round(4).to_string(index=False))
print(detail.round(4).to_string(index=False))

**Result: reproduced.** In all 30 responses, under every reading, more than half of the
local models have a complex eigenvalue among their top two (adopted: 0.87 to 1.00).

## fig5b-period-changes

> Over a response the orbital period may slow, speed up or not change; all three occur among the 30
> responses.

In [ ]:
t0 = time.time()
PC = {}
rows = []
for k in ["own/min80", "pooled/le80", "own/le80", "pooled/min80"]:
    out = []
    for n, o in enumerate(RUNS[k]):
        w = o["windows"][-1]
        hi = np.flatnonzero(w["dens"] >= 0.9)
        sl = slice(hi[0], hi[-1] + 1)
        wt = 1 / w["sddelay"]
        wt[~np.isfinite(wt)] = 0
        wt = wt * w["dens"]
        r, p, nw = apl.weighted_spearman(w["mid_t"][sl], w["mdelay"][sl], wt[sl], 10000, np.random.default_rng(500 + n))
        row = {"prep": o["prep"], "bout": o["bout"], "rho": r, "p": p, "n_windows": nw}
        if k == ADOPT:
            rc, pcode, _ = apl.weighted_spearman(w["mid_t"][sl], w["mdelay"][sl], wt[sl], 100000, np.random.default_rng(900 + n), pooled=True)
            row.update(p_code=pcode)
        out.append(row)
    X = pd.DataFrame(out)
    PC[k] = X
    rows.append({"reading": k, "slowing_p<0.01": int(((X.rho > 0) & (X.p < 0.01)).sum()),
                 "speeding_p<0.01": int(((X.rho < 0) & (X.p < 0.01)).sum()), "no_change": int((X.p >= 0.01).sum()),
                 "windows_min": X.n_windows.min(), "windows_max": X.n_windows.max()})
print(f"permutation tests in {time.time() - t0:.0f} s")
print(pd.DataFrame(rows).to_string(index=False))
X = PC[ADOPT]
print("\nadopted reading; p from 10000 permutations of the delays (two-sided), p_code from the code's 100000 pooled permutations (one-sided):")
print(X.round(4).to_string(index=False))
print("code test: slowing", int(((X.rho > 0) & (X.p_code < 0.01)).sum()), "speeding", int(((X.rho < 0) & (X.p_code < 0.01)).sum()),
      "no change", int((X.p_code >= 0.01).sum()))

**Result: reproduced.** Under the adopted reading, 12 responses slow (rho > 0, p < 0.01), 4
speed up (rho < 0, p < 0.01) and 14 do not change at that level; the code's pooled one-sided test
gives 13, 4 and 13. The number of windows per response is 31 to 73, within 27.9 to 79.2. The other
readings give 10 to 14 slowing, 4 to 6 speeding and 11 to 15 unchanged. The windows used run from the first to the last
window with at least 90% recurrent points.

## fig5c-no-relation-contraction

> There is no relationship between the change in orbital period over a response and its rate of
> contraction, as a stable spiral predicts.

In [ ]:
rows = []
for k, X in PC.items():
    E = EIG[k]
    r, p = stats.pearsonr(X.rho, E.a)
    contraction = np.exp(100 * E.a) * 100 - 100
    r2, p2 = stats.pearsonr(X.rho, contraction)
    rows.append({"reading": k, "R_rho_vs_a": r, "p": p, "R_rho_vs_contraction_pct": r2, "p_contraction": p2})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** Across the 30 responses, Pearson's R between the weighted rho and the
mean real part of the maximum eigenvalue is -0.341 (p = 0.065) under the adopted reading, and
-0.342 (p = 0.064) with the contraction in %/s. The other readings give R = -0.14 to -0.34 (p = 0.064
to 0.46). The test does not find the relation at p < 0.05, but the effect is moderate and p is close
to the threshold under two readings.

## fig6s1-no-p10-motorneurons

> No recorded neuron is a P10 motorneuron; no neuron comes close to one-to-one locking of its spikes
> with P10 spikes at any delay, nor locks consistently across programs.

In [ ]:
rows = []
for q in D["p10"]:
    r = next(r for r in recs if r["prep"] == q["prep"] and r["bout"] == q["bout"])
    for i, tr in enumerate(apl.neuron_trains(r)):
        v, n = apl.p10_locking(tr, q["t"])
        rows.append({"prep": q["prep"], "bout": q["bout"], "neuron": i + 1, "maxP": v, "spikes_after_33s": n})
MN = pd.DataFrame(rows)
print("neuron-program values:", len(MN), "; without spikes after 33 s:", int(MN.maxP.isna().sum()))
print("largest maximum locking:", round(MN.maxP.max(), 3), "; at or above 0.9:", int((MN.maxP >= 0.9).sum()))
two = MN[MN.maxP >= 0.9].groupby(["prep", "neuron"]).size()
print("neurons at or above 0.9 in two programs:", int((two >= 2).sum()))
print("largest values:")
print(MN.sort_values("maxP", ascending=False).head(8).round(3).to_string(index=False))
for c in (0.5, 0.7):
    print(f"neuron-program values at or above {c}:", int((MN.maxP >= c).sum()))
print("per preparation, max:", MN.groupby("prep").maxP.max().round(3).to_dict())

**Result: reproduced.** Of 1050 neuron-program values (1 neuron without spikes after 33 s),
the largest maximum locking is 0.522 (preparation 4, program 1, a neuron with 46 spikes); none
reaches 0.9, so none does so in two programs. Only 2 values reach 0.5 and none 0.7.

## fig6d-decode-p10

> P10 nerve activity can be fitted and forecast from the low-dimensional trajectory in all nine
> responses with P10 recordings, so the trajectory encodes muscle commands.

Adopted width (median ISI of the recording); PCs for 80% (smallest number reaching it). The
Methods' windowing (fit 40 s, forecast 10 s, step 1 s, from 33 s; histories 50 to 200 ms) is the
main reading; the code's (training on 50% of the series, 0.2 s steps, histories 50 to 250 ms) is a
variant. Every reading of width and count is run with the Methods' windowing. The 90% models of
claim fig6s2 use the Methods' windowing with the full history grid under own/min80 and own/le80, and
with histories of 50, 100, 150 and 200 ms under pooled/le80, to fit the time limit.

In [ ]:
t0 = time.time()
def p10_jobs(frac, windowing, hists, mode="own", rule="min80"):
    J = []
    for q in D["p10"]:
        i = next(i for i, r in enumerate(recs) if r["prep"] == q["prep"] and r["bout"] == q["bout"])
        J.append((recs[i], q["t"], float(sds[mode][i]), frac, windowing, hists, rule))
    return J
DEC = {("own/min80", "80", "methods"): apl.run_decoding(p10_jobs(0.8, "methods", apl.HISTORIES_METHODS)),
       ("own/min80", "80", "code"): apl.run_decoding(p10_jobs(0.8, "code", apl.HISTORIES_CODE)),
       ("own/min80", "90", "methods"): apl.run_decoding(p10_jobs(0.9, "methods", apl.HISTORIES_METHODS))}
for k in ["own/le80", "pooled/min80", "pooled/le80"]:
    mode, rule = k.split("/")
    DEC[(k, "80", "methods")] = apl.run_decoding(p10_jobs(0.8, "methods", apl.HISTORIES_METHODS, mode, rule))
# 90% models for claim fig6s2: the full history grid under the two readings with the recording's own
# width; under pooled/le80 four histories only (50 to 200 ms in 50 ms steps), to fit the time limit.
DEC[("own/le80", "90", "methods")] = apl.run_decoding(p10_jobs(0.9, "methods", apl.HISTORIES_METHODS, "own", "le80"))
DEC[("pooled/le80", "90", "methods")] = apl.run_decoding(p10_jobs(0.9, "methods", [5, 10, 15, 20], "pooled", "le80"))
print(f"decoding in {time.time() - t0:.0f} s")
def dec_table(res, hists=None):
    rows = []
    for o in res:
        if hists is None:
            R = np.array([w["R"] for w in o["windows"]]); MAE = np.array([w["MAE"] for w in o["windows"]]); best = o["best"]
        else:
            best = min(hists, key=lambda h: np.nanmean(o["by_history"][h][1]))
            R, MAE = o["by_history"][best]
        ok = np.isfinite(R)
        sem = np.nanstd(R, ddof=1) / np.sqrt(ok.sum())
        rows.append({"prep": o["prep"], "bout": o["bout"], "dims": o["d"], "history_ms": best * 10, "windows": len(R),
                     "undefined_R": int((~ok).sum()), "mean_R": np.nanmean(R), "mean_R_minus_2sem": np.nanmean(R) - 2 * sem,
                     "mean_R2": np.nanmean(R ** 2), "mean_MAE": np.nanmean(MAE),
                     "train_R": np.nanmean([w["R_train"] for w in o["windows"]]) if hists is None else np.nan})
    return pd.DataFrame(rows)
for name, res, h in [("own/min80, Methods windowing, 80%", DEC[("own/min80", "80", "methods")], None),
                     ("own/min80, variant: histories up to 100 ms", DEC[("own/min80", "80", "methods")], list(range(5, 11))),
                     ("own/min80, variant: code windowing", DEC[("own/min80", "80", "code")], None)]:
    T = dec_table(res, h)
    print(f"\n{name}: responses with mean R - 2 s.e.m. > 0: {int((T.mean_R_minus_2sem > 0).sum())} of 9; windows in all {T.windows.sum()}")
    print(T.round(3).to_string(index=False))
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    for name, h in [("Methods", None), ("up to 100 ms", list(range(5, 11)))]:
        T = dec_table(DEC[(k, "80", "methods")], h)
        f = T[(T.prep == 4) & (T.bout == 1)].iloc[0]
        rows.append({"reading": k, "histories": name, "dims": T.dims.tolist(), "pass": int((T.mean_R_minus_2sem > 0).sum()),
                     "mean_R_min": T.mean_R.min(), "mean_R_max": T.mean_R.max(),
                     "prep4_bout1_mean_R": f.mean_R, "prep4_bout1_bound": f.mean_R_minus_2sem})
print("\nevery reading of width and count, Methods windowing (responses passing of 9):")
print(pd.DataFrame(rows).round(3).to_string(index=False))

In [ ]:
# Preparation 4, bout 1 against a response that decodes well (preparation 4, bout 2): the P10 rate and
# the forecast of four of the Methods' windows, adopted reading and each program's best history.
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
for ax, bout in zip(axes, [1, 2]):
    q = next(q for q in D["p10"] if q["prep"] == 4 and q["bout"] == bout)
    i = next(i for i, r in enumerate(recs) if r["prep"] == 4 and r["bout"] == bout)
    o = next(o for o in DEC[("own/min80", "80", "methods")] if o["prep"] == 4 and o["bout"] == bout)
    P = apl.Program(recs[i], float(sds["own"][i]))
    pcs = P.proj[:, :o["d"]]
    rate = apl.sdf_matrix([q["t"]], float(sds["own"][i]), P.t_end)[P.ix_stim, 0]
    h = o["best"]
    ax.plot(P.stimbins, rate, color="0.6", lw=0.8, label="P10 rate")
    for s0 in [33.0, 43.0, 53.0, 63.0]:
        i0, i1 = np.searchsorted(P.stimbins, s0 - 1e-9), np.searchsorted(P.stimbins, s0 + 40 - 1e-9)
        rows_ = np.arange(max(i0, h), i1)
        b = apl.poisson_glm(apl.lagged(pcs, rows_, h), rate[rows_])
        rf = np.arange(i1, min(i1 + 1000, rate.size))
        fc = np.exp(np.clip(apl.lagged(pcs, rf, h) @ b, -30, 30))
        ax.plot(P.stimbins[rf], fc, lw=1, label=f"forecast after fit from {s0:.0f} s")
        print(f"prep 4 bout {bout}, window from {s0:.0f} s: forecast R {np.corrcoef(rate[rf], fc)[0, 1]:.3f}; "
              f"P10 rate range {rate[rf].min():.1f} to {rate[rf].max():.1f}, forecast range {fc.min():.1f} to {fc.max():.1f} spikes/s")
    ax.set_ylabel("spikes/s"); ax.set_title(f"preparation 4, bout {bout}"); ax.set_ylim(0, np.percentile(rate, 99.5) * 1.5)
axes[0].legend(fontsize=7, ncol=3); axes[1].set_xlabel("time (s)"); axes[1].set_xlim(70, 120)
plt.tight_layout(); plt.show()

In [ ]:
# Diverging forecasts: windows whose largest forecast exceeds 10 times the response's largest P10
# rate (reported, not judged). The linear predictor is clipped at 30, so a diverging forecast reaches
# about 1e13 spikes/s, and a few such windows dominate a mean MAE over windows.
def diverging(res, frac_label):
    rows = []
    for o in res:
        lim = 10 * o["rate_max"]
        allh = np.concatenate(list(o["fc_max_by_history"].values()))
        sel = o["fc_max_by_history"][o["best"]]
        rows.append({"model": frac_label, "prep": o["prep"], "bout": o["bout"], "history_ms": o["best"] * 10,
                     "diverging_selected": int((sel > lim).sum()), "windows": sel.size,
                     "largest_forecast_selected": sel.max(), "diverging_all_histories": int((allh > lim).sum()),
                     "windows_all_histories": allh.size})
    return pd.DataFrame(rows)
rows = []
for key, res in DEC.items():
    T = diverging(res, "/".join(key))
    rows.append({"reading": key[0], "variance": key[1], "windowing": key[2],
                 "responses_with_diverging_selected": int((T.diverging_selected > 0).sum()),
                 "diverging_selected": int(T.diverging_selected.sum()), "windows_selected": int(T.windows.sum()),
                 "largest_forecast_selected": T.largest_forecast_selected.max(),
                 "diverging_all_histories": int(T.diverging_all_histories.sum()),
                 "windows_all_histories": int(T.windows_all_histories.sum())})
print("windows with a forecast above 10 times the response's largest P10 rate, per decoding run:")
print(pd.DataFrame(rows).to_string(index=False))
print("\nown/min80, code windowing, 80%, per response:")
print(diverging(DEC[("own/min80", "80", "code")], "code").to_string(index=False))

# Sensitivity to rounding: preparation 4, bout 1, own/min80, code windowing, with the PCs perturbed
# by Gaussian noise of 1e-9 times their SD (seeded).
rng_div = np.random.default_rng(611)
i41 = next(i for i, r in enumerate(recs) if r["prep"] == 4 and r["bout"] == 1)
q41 = next(q for q in D["p10"] if q["prep"] == 4 and q["bout"] == 1)
P41 = apl.Program(recs[i41], float(sds["own"][i41]), "min80")
pcs41 = P41.proj[:, :apl.n_pcs(P41.ev, 0.8, "min80")]
rate41 = apl.sdf_matrix([q41["t"]], float(sds["own"][i41]), P41.t_end)[P41.ix_stim, 0]
rows = []
for h in [5, 15]:
    for name, X in [("as computed", pcs41), ("perturbed 1e-9 SD", pcs41 + 1e-9 * pcs41.std(0) * rng_div.standard_normal(pcs41.shape))]:
        o = apl.decode_p10(X, rate41, P41.stimbins, h, step_s=0.2, train_frac=0.5)
        M = np.array([w["MAE"] for w in o]); R = np.array([w["R"] for w in o]); F = np.array([w["fc_max"] for w in o])
        rows.append({"history_ms": h * 10, "pcs": name, "mean_MAE": M.mean(), "median_MAE": np.median(M),
                     "mean_R": np.nanmean(R), "diverging": int((F > 10 * rate41.max()).sum()), "windows": len(o)})
print("\npreparation 4, bout 1, own/min80, code windowing, sensitivity to a perturbation of the PCs:")
print(pd.DataFrame(rows).to_string(index=False))

**Result: not reproduced.** With the Methods' windowing, 8 of 9 responses have mean forecast R
minus 2 s.e.m. above 0 under every reading of width and count (adopted: mean R of the other eight
0.36 to 0.89). Response 1 of preparation 4 fails under all four readings (mean R -0.060, -0.120,
-0.080 and -0.123 for own/min80, own/le80, pooled/min80 and pooled/le80; bounds -0.127 to -0.204),
with histories up to 100 ms and with the code's windowing (mean R -0.102, bound -0.131). Its
training fits are as good as the others' (mean training R 0.94; 0.91 to 0.97 in all nine), but its
forecasts fail: in the four windows plotted the forecast R is -0.59 to 0.27, and the first forecast
runs at 56 to 133 spikes/s against a P10 rate of 0 to 24 spikes/s, while response 2 of the same
preparation gives 0.81 to 0.94. The decoding holds in eight of nine responses. The criterion
requires it in each of the nine, the statement says "in all nine", and the one failure holds under
every reading, history range and windowing run here, so the result is a failure of what the claim
says. Verification round 3 agreed with the change from partially reproduced. The Methods' windowing
gives 40 forecast windows per response (360 in all), the code's 174 per response (1566); the
paper's N = 173 is close to the code's count per response. Preparation 4's first response is the
one the paper describes as apparently chaotic (Figure 3, figure supplement 1B).

The decoding does not give the same numbers in every execution, as the forecasts diverge in some
windows and the selected history and the means over windows follow them. The numbers above and below
are from this execution. The copy reviewed in verification round 3 gave mean R -0.060, -0.127,
-0.060 and -0.119 for preparation 4, bout 1 (bounds -0.128 to -0.206) and -0.101 with the code's
windowing, and the same 8 of 9 under every reading; the pass counts have been the same in every
execution.

Here, over all histories, 462 to 627 of 5760 windows (80% models, Methods windowing, the four
readings) and 1614 of 7830 (code windowing) have a forecast above 10 times the response's largest
P10 rate, up to the clip of the linear predictor (about 1e13 spikes/s). Of the histories selected
for the 80% models, one window diverges, under pooled/min80 (largest forecast 1.2e9 spikes/s); the
other 80% runs have none (largest forecast 338 to 660 spikes/s). For preparation 4, bout 1 with the
code's windowing, perturbing the PCs by 1e-9 of their SD leaves the 50 ms model nearly unchanged
(mean MAE 13.27 and 13.32, mean R -0.102 and -0.101, no diverging window) but changes the 150 ms
model's mean MAE from 5.3e11 to 4.2e11 (74 and 78 diverging windows of 174). A mean MAE over
windows with diverging forecasts therefore depends on rounding, and so can the history selected:
the copy reviewed in round 3 selected a history for this response with a mean MAE of 7.6e6, where
this execution's has 13.3. The forecast R of the selected models, on which the criterion rests,
moves by up to 0.02 between executions and the failure of this response holds in each.

## fig6s2-dimensions-p10

> The nine recordings with P10 activity need only 3 to 5 embedding dimensions for 80% variance, and
> at least double that for 90%; using the 90% dimensions does not improve the P10 forecasts.

In [ ]:
p10_idx = [next(i for i, r in enumerate(recs) if r["prep"] == q["prep"] and r["bout"] == q["bout"]) for q in D["p10"]]
rows = []
for mode in ["sqrt12", "twelve", "own", "pooled"]:
    evs = [apl.Program(recs[i], sds[mode][i]).ev for i in p10_idx]
    for rule in ["min80", "le80"]:
        d80 = np.array([apl.n_pcs(ev, 0.8, rule) for ev in evs])
        d90 = np.array([apl.n_pcs(ev, 0.9, rule) for ev in evs])
        rows.append({"sd": mode, "count": rule, "d80": d80.tolist(), "d90": d90.tolist(),
                     "all_in_2.7_to_5.5": bool(np.all((d80 >= 2.7) & (d80 <= 5.5))), "d90_at_least_2x": bool(np.all(d90 >= 2 * d80))})
print(pd.DataFrame(rows).to_string(index=False))
FOUR = [5, 10, 15, 20]
summ = []
for k in ["own/min80", "own/le80", "pooled/le80"]:
    full90 = len(DEC[(k, "90", "methods")][0]["by_history"]) == len(apl.HISTORIES_METHODS)
    for grid in (["full grid", "four histories"] if full90 else ["four histories"]):
        h = None if grid == "full grid" else FOUR
        T80, T90 = dec_table(DEC[(k, "80", "methods")], h), dec_table(DEC[(k, "90", "methods")], h)
        cmp = pd.DataFrame({"prep": T80.prep, "bout": T80.bout, "dims80": T80.dims, "dims90": T90.dims,
                            "hist80_ms": T80.history_ms, "hist90_ms": T90.history_ms,
                            "MAE80": T80.mean_MAE, "MAE90": T90.mean_MAE, "R2_80": T80.mean_R2, "R2_90": T90.mean_R2})
        w_mae = stats.wilcoxon(cmp.MAE90, cmp.MAE80)
        w_r2 = stats.wilcoxon(cmp.R2_90, cmp.R2_80)
        summ.append({"reading": k, "histories": grid, "MAE_p": w_mae.pvalue, "MAE90_lower": int((cmp.MAE90 < cmp.MAE80).sum()),
                     "MAE_median_diff": float(np.median(cmp.MAE90 - cmp.MAE80)), "R2_p": w_r2.pvalue,
                     "R2_90_higher": int((cmp.R2_90 > cmp.R2_80).sum()), "R2_median_diff": float(np.median(cmp.R2_90 - cmp.R2_80)),
                     "90_better_at_p<0.05": bool((w_mae.pvalue < 0.05 and np.median(cmp.MAE90 - cmp.MAE80) < 0) or
                                                 (w_r2.pvalue < 0.05 and np.median(cmp.R2_90 - cmp.R2_80) > 0))})
        if grid == "full grid":
            print(f"\n{k}, full history grid for both:")
            print(cmp.round(3).to_string(index=False))
print("\ntwo-sided Wilcoxon signed-rank tests, 90% against 80% models (MAE90_lower and R2_90_higher count responses of 9):")
print(pd.DataFrame(summ).round(4).to_string(index=False))

**Result: reproduced, under own/le80.** Under the readings with the code's count rule the nine
80% counts lie in 3 to 5 (own/le80: 3 to 5; pooled/le80: 3 to 5), and the 90% counts are at least
twice them in each. Under own/le80, with the full history grid for both, the 90% models are not
better: mean MAE lower with 90% in 5 of 9 (two-sided Wilcoxon p = 0.91, median difference -0.13) and
mean R-squared higher in 4 of 9 (p = 0.50, median difference -0.008). So one reading shows both parts.
Under pooled/le80 (four histories) the 90% models are not better either (p = 0.65 and 0.82). Under
the adopted reading (own/min80) the counts are 4 to 6, and one (6) is outside 2.7 to 5.5; there the
90% models are worse in MAE (p = 0.027, 7 of 9 worse, median +1.23 with the full grid; p = 0.020 with
four histories for both) and do not differ in R-squared (p = 0.13). No test finds the 90% models
better at p < 0.05 in any of the five comparisons. Of the selected 90% models one, two and two windows
diverge under own/min80, own/le80 and pooled/le80 (largest forecast 2.5e6, 1.4e7 and 8.1e8 spikes/s),
and over all histories 1166 and 1111 of 5760 windows diverge under own/min80 and own/le80 (fig6d's
printout), so the mean MAEs and these p-values depend on rounding through the history selected and
differ between executions: this execution gives the numbers above, and earlier executions gave
p = 0.50 and 0.91 for MAE under own/le80, with the same conclusion each time.

## Participation

Per program, from its own PCA under the adopted width; d is (a) the program's own count for 80%
(Methods) or (b) the count from the PCA of the three programs concatenated, at most 80%, used for
all three (code). Normalised to the program's maximum.

In [ ]:
progs = [apl.Program(r, s, "min80") for r, s in zip(recs, sds["own"])]
PART = {}
for reading in ["a_own", "b_common"]:
    for p in range(1, 11):
        ix = sorted([i for i, r in enumerate(recs) if r["prep"] == p], key=lambda i: recs[i]["bout"])
        if reading == "b_common":
            _, _, evc = apl.pca(np.vstack([progs[i].sdf[progs[i].ix_stim] for i in ix]))
            ds = [apl.n_pcs(evc, 0.8, "le80")] * 3
        else:
            ds = [apl.n_pcs(progs[i].ev, 0.8, "min80") for i in ix]
        PART[(reading, p)] = (np.array([apl.participation(progs[i].coeffs, progs[i].ev, d)[1] for i, d in zip(ix, ds)]), ds)
print({k: v[1] for k, v in PART.items()})

## fig7a-long-tailed-participation

> Every response has a long-tailed distribution of single-neuron participation, a minority of
> neurons dominating its dynamics.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    for b in range(3):
        rows.append({"reading": reading, "prep": p, "bout": b + 1, "skewness": stats.skew(P[b]), "frac_over_half": np.mean(P[b] > 0.5),
                     "median": np.median(P[b])})
PA = pd.DataFrame(rows)
print(PA.groupby("reading").apply(lambda d: pd.Series({"skew_positive": int((d.skewness > 0).sum()), "skew_min": d.skewness.min(),
      "frac_over_half_below_0.5": int((d.frac_over_half < 0.5).sum()), "frac_over_half_max": d.frac_over_half.max(),
      "median_participation_median": d["median"].median()})).round(3).to_string())
print(PA[PA.reading == "a_own"].round(3).to_string(index=False))

**Result: reproduced.** Under both readings of d, every one of the 30 responses has positive
skewness of participation (0.50 to 4.36, reading (a)) and fewer than half its neurons above 50% of
the maximum (at most 39.4%).

## fig7f-strongly-variable-neurons

> Neurons with high participation in one response often have low participation in another; every
> preparation contains neurons whose change in participation well exceeds that predicted by a
> noise model.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    dP = np.column_stack([P[0] - P[1], P[1] - P[2], P[0] - P[2]])
    mx = np.abs(dP[:, :2]).max(1)
    for pool in ["all_pairs", "consecutive"]:
        data = dP.ravel(order="F") if pool == "all_pairs" else dP[:, :2].ravel(order="F")
        mu, sd, nout = apl.iterate_outliers(data)
        thr = max(abs(mu - 3 * sd), abs(mu + 3 * sd))
        nv = int((mx >= thr).sum())
        rows.append({"reading": reading, "pool": pool, "prep": p, "neurons": P.shape[1], "mu": mu, "sd": sd, "dropped": nout,
                     "threshold": thr, "strongly_variable": nv, "proportion": nv / P.shape[1]})
SV = pd.DataFrame(rows)
print(SV.groupby(["reading", "pool"]).apply(lambda d: pd.Series({"preps_with_one": int((d.strongly_variable > 0).sum()),
      "min_count": d.strongly_variable.min(), "proportion_range": f"{d.proportion.min():.3f} to {d.proportion.max():.3f}"})).to_string())
print(SV.round(4).to_string(index=False))

**Result: reproduced.** Under both readings of d and both poolings, all 10 preparations have
strongly variable neurons: at least 4 per preparation (reading (a), all pairs), 3.8% to 23.4% of the
neurons; with consecutive pairs (the code) 5.9% to 29.3%.

## halving-participation

> Even the most strongly participating neurons in a response could more than halve their
> participation in other evoked responses.

In [ ]:
rows = []
for (reading, p), (P, ds) in PART.items():
    for sel_name in ["top_neuron", "top_10pct"]:
        hit, examples = False, []
        for a in range(3):
            sel = [int(np.argmax(P[a]))] if sel_name == "top_neuron" else list(np.argsort(P[a])[::-1][:int(np.ceil(0.1 * P.shape[1]))])
            for b in range(3):
                if b == a:
                    continue
                lo = [n for n in sel if P[b][n] < 0.5 * P[a][n]]
                if lo:
                    hit = True
                    examples.append((a + 1, b + 1, len(lo), round(float(min(P[b][n] / P[a][n] for n in lo)), 3)))
        rows.append({"reading": reading, "selection": sel_name, "prep": p, "halves": hit, "examples": examples[:3]})
HV = pd.DataFrame(rows)
print(HV.groupby(["reading", "selection"]).halves.sum().to_string())
print(HV[HV.reading == "a_own"].to_string(index=False))

**Result: reproduced.** The neuron at 100% in a program has under half that participation in
another program of its preparation in 6 of 10 preparations (reading (a)); a neuron of a program's
top 10% does so in 9 of 10. Both readings of d give the same counts.

## fig7s1-rate-synchrony

> A neuron's participation reflects both its firing rate and its synchrony; firing rate explains on
> average 50% of the variation in participation within a response, and rate changes 23% of the
> change in participation between responses.

In [ ]:
rows = []
for reading in ["a_own", "b_common"]:
    for rate_from in [32.5, 30.0]:
        r2, dr2, dr2_code = [], [], []
        for p in range(1, 11):
            ix = sorted([i for i, r in enumerate(recs) if r["prep"] == p], key=lambda i: recs[i]["bout"])
            P = PART[(reading, p)][0]
            rates = np.array([[np.sum(tr >= rate_from) / (progs[i].t_end - rate_from) for tr in progs[i].trains] for i in ix])
            r2 += [np.corrcoef(P[b], rates[b])[0, 1] ** 2 for b in range(3)]
            dR = np.column_stack([rates[0] - rates[1], rates[0] - rates[2], rates[1] - rates[2]])
            dPm = np.column_stack([P[0] - P[1], P[0] - P[2], P[1] - P[2]])
            dPc = np.column_stack([P[0] - P[1], P[1] - P[2], P[0] - P[2]])
            dr2 += [np.corrcoef(dR[:, c], dPm[:, c])[0, 1] ** 2 for c in range(3)]
            dr2_code += [np.corrcoef(dR[:, c], dPc[:, c])[0, 1] ** 2 for c in range(3)]
        rows.append({"reading": reading, "rate_from_s": rate_from, "mean_R2_within": np.mean(r2), "range_within": f"{min(r2):.2f} to {max(r2):.2f}",
                     "mean_R2_change_matched": np.mean(dr2), "mean_R2_change_code_pairing": np.mean(dr2_code), "n_programs": len(r2), "n_pairs": len(dr2)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** With participation reading (a), the rate from stimulation offset and
matched pairs, the mean R-squared is 0.491 within responses (0.45 to 0.55 required; range 0.29 to
0.74) and 0.239 for changes between responses (0.207 to 0.253 required). The rate from onset gives
0.514 and 0.265 (the second outside); the code's mismatched pairing gives 0.205 and 0.221; reading
(b) gives 0.499 and 0.244.

## fig7gh-robust-to-participation

> The distance between a pair of responses does not correlate with the total change in neuron
> participation between them nor with the distance between their participation distributions; the
> program is robust to the participation of single neurons.

In [ ]:
# Participation under each reading of width and count: (a) the program's own count under the rule,
# (b) the common count (at most 80% of the concatenated PCA), as in the Participation section.
PARTR = {}
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    mode, rule = k.split("/")
    pg = progs if mode == "own" else [apl.Program(r, s, "min80") for r, s in zip(recs, sds[mode])]
    for p in range(1, 11):
        ix = sorted([i for i, r in enumerate(recs) if r["prep"] == p], key=lambda i: recs[i]["bout"])
        _, _, evc = apl.pca(np.vstack([pg[i].sdf[pg[i].ix_stim] for i in ix]))
        for reading, ds in [("a_own", [apl.n_pcs(pg[i].ev, 0.8, rule) for i in ix]), ("b_common", [apl.n_pcs(evc, 0.8, "le80")] * 3)]:
            PARTR[(k, reading, p)] = np.array([apl.participation(pg[i].coeffs, pg[i].ev, d)[1] for i, d in zip(ix, ds)])
rows = []
for k in ["own/min80", "own/le80", "pooled/min80", "pooled/le80"]:
    for ps in ["recurrent", "code"]:
        T = haus_table(k, "prog1", ps)
        T["norm_dist"] = T.H / T.ctrl_mean
        for reading in ["a_own", "b_common"]:
            tc, hd = [], []
            for row in T.itertuples():
                P = PARTR[(k, reading, row.prep)]
                i, j = [int(x) - 1 for x in row.pair.split(",")]
                tc.append(np.abs(P[i] - P[j]).sum())
                hd.append(apl.hellinger(P[i], P[j]))
            T["total_change"], T["hellinger"] = tc, hd
            keep = ~((T.prep == 4) & T.pair.isin(["1,2", "1,3"]))
            for x in ["total_change", "hellinger"]:
                X = sm.add_constant(T.loc[keep, x]); y = T.loc[keep, "norm_dist"]
                ols = sm.OLS(y, X).fit()
                rob = sm.RLM(y, X, M=sm.robust.norms.TukeyBiweight()).fit()
                ck = T.norm_dist < 1
                ols_code = sm.OLS(T.loc[ck, "norm_dist"], sm.add_constant(T.loc[ck, x])).fit()
                rows.append({"reading": k, "points": ps, "participation": reading, "x": x, "n": int(keep.sum()),
                             "slope": ols.params.iloc[1], "p": ols.pvalues.iloc[1], "R2": ols.rsquared,
                             "robust_slope": rob.params.iloc[1], "robust_p": rob.pvalues.iloc[1],
                             "code_exclusion_n": int(ck.sum()), "code_exclusion_p": ols_code.pvalues.iloc[1]})
RB = pd.DataFrame(rows)
print(RB.round(4).to_string(index=False))
V = RB.groupby(["reading", "points", "participation"]).p.apply(lambda s: int((s < 0.05).sum())).rename("significant_slopes_of_2").reset_index()
print("\nslopes significant at p < 0.05 (of the two) per reading, point set and participation reading:")
print(V.pivot_table(index=["reading", "points"], columns="participation", values="significant_slopes_of_2").to_string())

**Result: partially reproduced, resting on the adopted reading.** Under the adopted width and
count, each set of distances finds one significant slope of two: with the recurrent points
(Methods), the normalised distance rises with the total change in participation (slope 0.0127,
p = 0.031, R-squared 0.17, 28 pairs) but not with the Hellinger distance (p = 0.35); with the code's
point sets, it rises with the Hellinger distance (p = 0.028) but not with the total change (p =
0.11). Participation reading (b) gives the same pattern (p = 0.032 and 0.017). So one of the two
absences holds and the other fails. Under own/le80 one slope is significant with the recurrent
points (total change, p = 0.036) and none with the code's points (lowest p 0.053); under pooled/min80
and pooled/le80, with participation from the pooled width, no slope is significant with the legend's exclusion (lowest p 0.085
and 0.069). With the code's exclusion (pairs with normalised distance below 1, 29 pairs with the
recurrent points), pooled/min80 has a significant total-change slope with the recurrent points
(p = 0.028 and 0.029, participation readings (a) and (b)), and pooled/le80 has none (lowest p 0.069).
The outcome depends on the reading, and under pooled/min80 on the exclusion rule.

## fig7s2-no-invariant-core

> No small invariant core of strongly participating, phasically active neurons, such as a classic
> central pattern generator, is found across preparations.

Ensembles by the code's consensus clustering of each program from stimulation onset, typed by the
autocorrelogram of their pooled spikes against interval shuffles (port in `apl.py`, citing each
MATLAB file). A neuron takes its ensemble's type; persistent neurons have the same type in all three
programs. Participation is that of the Participation section (readings (a) and (b)). Run for the
recording's own width (the code's Data.GaussSD, the adopted width) and the pooled width, each with two
seeds.

In [ ]:
t0 = time.time()
TYPES = {(mode, seed): apl.run_types(recs, sds[mode], seed=seed) for mode in ["own", "pooled"] for seed in [7, 8]}
print(f"ensembles and types of 30 recordings, 2 widths x 2 seeds, in {time.time() - t0:.0f} s")
names = {1: "tonic", 2: "oscillator", 3: "burster", 4: "pauser"}
rows = []
for (mode, seed), res in TYPES.items():
    for o in res:
        rows.append({"width": mode, "seed": seed, "prep": o["prep"], "bout": o["bout"], "ensembles": len(o["groups"]),
                     "clustered": o["n_clustered"], **{names[t]: int((o["types"] == t).sum()) for t in names}})
ET = pd.DataFrame(rows)
print("\nneurons per type (sum over 30 programs) and ensembles per program:")
print(ET.groupby(["width", "seed"])[["ensembles", "clustered", "tonic", "oscillator", "burster", "pauser"]]
      .agg(["sum", "min", "max"]).to_string())
print("programs that could not be grouped:", {k: sum(o["ungrouped"] for o in v) for k, v in TYPES.items()})

def core_table(res, reading):
    out = []
    for p in range(1, 11):
        T3 = np.column_stack([o["types"] for o in sorted([o for o in res if o["prep"] == p], key=lambda o: o["bout"])])
        P = PART[(reading, p)][0]
        high, dmax = P.max(0), np.abs(np.column_stack([P[0] - P[1], P[1] - P[2]])).max(1)
        persist = {names[t]: int(np.all(T3 == t, 1).sum()) for t in names}
        osc = np.flatnonzero(np.all(T3 == 2, 1))
        if osc.size:
            upper = np.percentile(high[osc], 90, method="hazen")     # MATLAB prctile
            sel = osc[high[osc] > upper]
        else:
            sel = osc
        med = float(np.median(dmax[sel])) if sel.size else np.nan
        top_all = np.argsort(high)[::-1][:int(np.ceil(0.1 * high.size))]
        out.append({"prep": p, "neurons": high.size, **{f"persistent_{k}": v for k, v in persist.items()},
                    "top10pct_osc_n": int(sel.size), "top_osc_median_maxP": float(np.median(high[sel])) if sel.size else np.nan,
                    "top_osc_median_max_change": med, "core": bool(med < 0.1),
                    "no_typing_top10pct_median_max_change": float(np.median(dmax[top_all]))})
    return pd.DataFrame(out)

summary = []
for (mode, seed), res in TYPES.items():
    for reading in ["a_own", "b_common"]:
        CT = core_table(res, reading)
        summary.append({"width": mode, "seed": seed, "participation": reading,
                        "preps_with_core": int(CT.core.sum()), "preps_without_persistent_oscillator": int((CT.persistent_oscillator == 0).sum()),
                        "median_change_min": CT.top_osc_median_max_change.min(), "median_change_max": CT.top_osc_median_max_change.max(),
                        **{f"preps_with_persistent_{k}": int((CT[f'persistent_{k}'] > 0).sum()) for k in names.values()},
                        "no_typing_preps_under_0.1": int((CT.no_typing_top10pct_median_max_change < 0.1).sum()),
                        "no_typing_min": CT.no_typing_top10pct_median_max_change.min()})
        if (mode, seed, reading) == ("own", 7, "a_own"):
            print("\nadopted width (own), seed 7, participation (a), per preparation:")
            print(CT.round(3).to_string(index=False))
print("\nall variants (core: median maximum change of the top 10% of persistent oscillators under 0.1):")
print(pd.DataFrame(summary).round(3).to_string(index=False))

**Result: reproduced.** With the port of the code's ensemble toolbox, every preparation has
neurons typed oscillator in all three programs (16 to 84 per preparation, adopted width and seed),
and the top 10% of them by maximum participation have a median maximum change between consecutive
programs of 0.068 to 0.440 of the program maximum. Only preparation 2 has an invariant core by the
claims file's rule (median change 0.068, under 0.1); 1 of 10 preparations, fewer than half, under
the adopted width with seed 7, and 0 or 1 of 10 under both widths, both seeds and both readings of
participation. The top 10% of persistent oscillators holds 2 to 8 neurons per preparation. Without
typing, the top 10% of all neurons by maximum participation have median maximum changes of 0.12 to
0.32, under 0.1 in no preparation. The legend's statement that only the oscillator class had a
consistently typed neuron in all ten preparations holds in part: oscillators persist in 10 of 10
preparations under every variant, but persistent bursters are found in 6 to 10 preparations (10 in
one variant), persistent tonic neurons in 2 or 3 and persistent pausers in none. The clustering is
random (k-means starts); two seeds are shown, and the port has not been checked against MATLAB's
output.

## fig8-caudolateral

> Neurons with strong participation across the three responses, and those with highly variable
> participation, lie mainly in the caudo-lateral quadrant of the ganglion.

Blocked (see the header): neither the data set, its documentation nor the paper's text says which
directions of the array's x and y axes are caudal and lateral. The counts by quadrant of the array
coordinates are shown so that the claim can be judged once the orientation is known, with true
quintiles as the criterion's variant, and the share of the top fifth in its most common quadrant
against the share of all neurons there; they decide nothing here.

In [ ]:
rows = []
for side, group in [("left", [1, 2, 4, 10]), ("right", [3, 5, 6, 9])]:
    vals_max, vals_rng, xs, ys = [], [], [], []
    for p in group:
        P = PART[("a_own", p)][0]
        vals_max.append(P.max(0)); vals_rng.append(P.max(0) - P.min(0))
        xs.append(D["coords"][p - 1]["x"]); ys.append(D["coords"][p - 1]["y"])
    allmax = np.concatenate([PART[("a_own", p)][0].max(0) for p in [1, 2, 3, 4, 5, 6, 9, 10]])
    allrng = np.concatenate([np.ptp(PART[("a_own", p)][0], 0) for p in [1, 2, 3, 4, 5, 6, 9, 10]])
    x, y = np.concatenate(xs), np.concatenate(ys)
    cx, cy = 25.0, 25.0     # centre of the 501 x 501 template, in coordinate units
    q = np.where(x >= cx, "x+", "x-").astype(object) + np.where(y >= cy, "y+", "y-")
    for name, v, allv in [("max", np.concatenate(vals_max), allmax), ("range", np.concatenate(vals_rng), allrng)]:
        edges = np.linspace(allv.min(), allv.max(), 6)
        for division, top in [("equal-width fifth (code)", v >= edges[-2]), ("true quintile (variant)", v >= np.quantile(v, 0.8))]:
            counts = pd.Series(q[top]).value_counts()
            rows.append({"side": side, "measure": name, "division": division, "neurons": v.size, "top_fifth": int(top.sum()),
                         "by_quadrant": counts.to_dict(), "largest_quadrant": counts.index[0] if top.any() else None,
                         "largest_share": round(counts.iloc[0] / top.sum(), 2) if top.any() else np.nan,
                         "all_neurons_in_that_quadrant": round(float(np.mean(q == counts.index[0])), 2) if top.any() else np.nan})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked, metadata insufficient** (see the header). Shown for the verifier, not judged:
in each side the top-fifth neurons concentrate in one quadrant of the array coordinates, the same
quadrant for maximum under both divisions and for range under true quintiles (left x-y-, right
x-y+). Under true quintiles
that quadrant holds 34% to 41% of the top fifth against 19% to 20% of all neurons; under the code's
equal-width fifths 71% (left, maximum, 15 of 21) and 54% (right, 15 of 28). The code's equal-width
division leaves only 1 or 2 neurons in the top fifth of the range, so that half of the test is
nearly empty as the code defines it. Which anatomical quadrant these are cannot be told without the
array's orientation.

## Summary

| Claim | Outcome |
|---|---|
| dataset-size | partially reproduced |
| fig1d-slow-oscillation | reproduced |
| fig1e-decaying-rate | reproduced |
| fig2b-dimensions | reproduced (under the code's count rule) |
| fig2ef-periodic-recurrence | reproduced |
| fig2e-single-dominant-orbit | reproduced |
| fig2s2-threshold-robust | reproduced |
| fig3a-coalescence | not reproduced |
| fig3c-same-manifold | reproduced |
| fig3s1a-common-axes | reproduced |
| fig3d-initial-conditions | reproduced |
| fig3f-pairwise-similarity | reproduced |
| fig3h-perturbations-return | partially reproduced (assumed) |
| fig3i-same-manifold-after | reproduced (reading (b)) |
| fig4bc-stable-spiral | partially reproduced |
| fig4c-orbital-period | reproduced (assumed) |
| fig4s1a-contracting-dimensions | not reproduced |
| fig4s1b-rotation-dominant | reproduced |
| fig5b-period-changes | reproduced |
| fig5c-no-relation-contraction | reproduced |
| fig6s1-no-p10-motorneurons | reproduced |
| fig6d-decode-p10 | not reproduced |
| fig6s2-dimensions-p10 | reproduced (under own/le80) |
| fig7a-long-tailed-participation | reproduced |
| fig7f-strongly-variable-neurons | reproduced |
| halving-participation | reproduced |
| fig7s1-rate-synchrony | reproduced |
| fig7gh-robust-to-participation | partially reproduced (assumed) |
| fig7s2-no-invariant-core | reproduced |
| fig8-caudolateral | blocked, metadata insufficient |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**Outcome under each reading of width and count**, for the claims whose result changes with it
(numbers printed in each claim's section; "holds" means the criterion is met under that reading):

| Claim | own/min80 (adopted) | own/le80 | pooled/min80 | pooled/le80 |
|---|---|---|---|---|
| fig2b-dimensions | fails (ratio 9.4 in one response) | holds | fails (ratio 9.4) | holds |
| fig3h-perturbations-return, mean period | fails (16 kept, 15 returned) | holds (18, 16) | fails (14, 13) | fails (16, 14) |
| fig3h-perturbations-return, median period | fails (17, 16) | holds (19, 17) | fails (14, 13) | fails (17, 15) |
| fig3i-same-manifold-after, (b), mean period | holds (13) | fails (16) | fails (11) | holds (14) |
| fig4c-orbital-period | holds (10.45 s) | fails (13.55 s) | holds (10.12 s) | fails (12.22 s) |
| fig6s2-dimensions-p10 | fails (one count of 6) | holds | fails (counts to 6) | holds |
| fig6d-decode-p10 | 8 of 9 | 8 of 9 | 8 of 9 | 8 of 9 |
| fig7gh-robust-to-participation | one slope of two significant | one slope with recurrent points | no slope (one with the code's exclusion) | no slope |

No reading makes all of these hold. The adopted reading follows the Methods' count and the
recording's own median ISI; the outcomes that rest on it are marked as assumed.

**How the analysis differs from the paper's.** It is a Python port of the paper's MATLAB code,
following the Methods where they fix a detail and the code where they do not. The Methods' kernel
width as printed gives 21 to 76 dimensions (33 to 76 with median ISI / 12, 21 to 41 with / sqrt(12)), and the analysis uses the code's width
(the median ISI) for everything after claim fig2b. The code computes that median over the recordings
of the same bout number read so far; the adopted reading uses the recording's own. The number of PCs
is the smallest reaching 80% (Methods) in the adopted reading; the code's rule (at most 80%) is run
alongside and changes some outcomes (table above). A window counts as coalesced or recurrent at at
least 90% of its points, as the Methods say. The local models use the contiguous run of neighbours
as the Methods describe. The permutation test of fig5b permutes the delays against time (10000
permutations); the code's test is shown too. The decoding uses a numpy IRLS for MATLAB's glmfit; the
90% models use the full history grid under the two readings with the recording's own width and four
histories under pooled/le80. The ensembles and types of fig7s2 come from a Python port of the code's
consensus clustering and autocorrelogram typing, with Lloyd's k-means for MATLAB's kmeans, not
checked against MATLAB's output.

**What was found about the data set.** The files match their checksums and the description: 30
recordings of 75 to 160 neurons, 1131 in all, P10 for nine, coordinates for all ten preparations.
The paper's 120 to 180 neurons per recording cannot be right with its own total of 1131. The P10
sampling (1629.7 Hz) agrees with the description's 1630 Hz, not the paper's 1600 Hz. The P10 file
names of preparation 2 drop the "B" of its spike file. The coordinates have no unit and no
orientation, so which part of the array is caudo-lateral cannot be told from the data set.

In [ ]:
print(f"notebook run time {(time.time() - started) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB (main process)")